In [ ]:


import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from confidence.learned.deepsvd import DeepSVDD
from model.classifier import Classifier, MyProgressBar
from utils.transforms.apply import grid_resample

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from its.transform import multi_transform
from utils.affine_transforms import AffineTransformation2D
from dataset.mnist_no_pil import NoPILMNIST, NoPILFashionMNIST, AffineTransformDataset

dataset = 'mnist' # or 'fmnist'
batch_size = 128
#divide by 255 transform


if dataset == 'mnist':
    # essentially excluding either 6 or 9 as these would cause degenerations under 180 degree rotation
    dataset_train = NoPILMNIST("../experiment_files/data",train=True,download=True)
    n_classes = 10
    dataset_val_test = NoPILMNIST("../experiment_files/data",train=False,download=True)
    #split into validation and test set
    dataset_val, dataset_test_pre = #fix this, use val from train as this is what is usually expected.
elif dataset == 'fmnist':
    dataset_train = NoPILFashionMNIST("../experiment_files/data",train=True,download=True)
    n_classes = 10
    dataset_val_test = NoPILFashionMNIST("../experiment_files/data",train=False,download=True)
    #split into validation and test set
    dataset_val, dataset_test_pre = #fix this, use val from train as this is what is usually expected.



transformations = [AffineTransformation2D.ROTATION.value,AffineTransformation2D.SHEARING_X.value,AffineTransformation2D.SHEARING_Y.value
                     ,AffineTransformation2D.SCALING_X.value,AffineTransformation2D.SCALING_Y.value]
domains = [(-torch.pi,torch.pi), ((-0.5, 0.5),), ((-0.5, 0.5),), ((1/1.3-1, 0.3),), ((1/1.3-1, 0.3),)]
n_samples = 17

def transform(batch_size):
    n = torch.randint(0, n_samples, (batch_size, len(transformations)), device="cpu")
    x_test = torch.zeros(batch_size, 1, 28, 28, device="cpu")
    _, _, T = multi_transform(x_test, transformations, n, n_samples=n_samples, domain=domains)
    return T


dataset_test = AffineTransformDataset(dataset_test_pre,transform,return_transformation=False,batch_size=batch_size)


In [ ]:
dataset_train_transformed = AffineTransformDataset(dataset_train,transform,return_transformation=False,batch_size=batch_size)


In [ ]:
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=4,persistent_workers=True)
val_loader = torch.utils.data.DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=4,persistent_workers=True)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size, shuffle=False, num_workers=4,persistent_workers=True)

In [ ]:
train_loader_transformed = torch.utils.data.DataLoader(dataset_train_transformed, batch_size=batch_size, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
#test images
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 5))

axs[0].imshow(torchvision.utils.make_grid(next(iter(train_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(next(iter(val_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[2].imshow(torchvision.utils.make_grid(next(iter(test_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Training')
axs[1].set_title('Validation')
axs[2].set_title('Test')

for ax in axs.flat:
    ax.axis('off')

In [ ]:
import pytorch_lightning as pl
import os

model_path = f'../model/{dataset}.pth'

model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),  # Output: 32x28x28
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 32x14x14
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # Output: 64x14x14
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),                  # Output: 64x7x7
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)


# Check if model is already trained
if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model.load_state_dict(torch.load(model_path))
else:
    print(f"Training model and will save to {model_path}")
    lightning_model = Classifier(model, optimizer_class =  torch.optim.AdamW, optimizer_params = {"lr": 1e-3})
    progress_bar = MyProgressBar()
    trainer = pl.Trainer(
        accelerator="cuda",
        max_epochs=10,
        precision="16-mixed",
        callbacks=[progress_bar],
    )
    # Train the model
    trainer.fit(lightning_model, train_loader,val_loader)
    # Test the model
    #trainer.test(lightning_model, test_loader)
    # Save model
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")

In [ ]:
#calibrate the model
#from confidence.calibration import TemperatureCalibrationModule
#if model is not temperature calibrated
#if isinstance(model, TemperatureCalibrationModule):
    #model = model.model
#model = TemperatureCalibrationModule(model,per_logit=True).cuda()
#model.fit(test_loader)


In [ ]:
model.cuda()

In [ ]:
with torch.no_grad():
    model.eval()
    test_acc = 0
    for data, target in val_loader:
        data, target = data.cuda(), target.cuda()
        output = model(data)
        test_acc += output.argmax(dim=-1).eq(target).sum().item()
    test_acc /= len(dataset_val)
    print(f'Mean accuracy on the val set: {test_acc}.')

In [ ]:
with torch.no_grad():
    model.eval()
    test_acc = 0
    for data, target in test_loader:
        data, target = data.cuda(), target.cuda()
        output = model(data)
        test_acc += output.argmax(dim=-1).eq(target).sum().item()
    test_acc /= len(dataset_test)
    print(f'Mean accuracy on the transformed test set: {test_acc}.')

In [ ]:
from confidence.utils import ModelInputWrapper

#use model wrapper to extract the embeddings
dual_ouput_model = ModelInputWrapper(model,'9',flatten=True)

In [ ]:
#extract embeddings and fit NNDistanceConfidence()
embeddings=[]
classes =[]
for batch in train_loader:
    emb,_ = dual_ouput_model(batch[0].cuda())
    embeddings.append(emb.detach().cpu().numpy())
    classes.append(batch[1].detach().cpu().numpy())


embeddings = np.vstack(embeddings)
classes = np.hstack(classes)

#randomly sample 5000 ones
if embeddings.shape[0] > 5000:
    indices = np.random.choice(embeddings.shape[0], 5000, replace=False)
    embeddings_sampled = embeddings[indices]
    classes_sampled = classes[indices]

In [ ]:
#test simulated annealing
from utils.transformation_problem import TransformationProblem
from utils.transform_sequence import TransformSequence
transform_seq= TransformSequence(transformations, domains, neighbour_hood_size =0.25,application_method=grid_resample,device=device,reflect=True,use_individual_param_correction=False)





In [ ]:
import search.shgo
import importlib
importlib.reload(search.shgo)

In [ ]:
di = search.shgo.SHGO(selection_method="topk")
di_no_grad = search.shgo.SHGO(selection_method="topk",local_max_steps=0)

In [ ]:
from confidence.base_confidence import SplitConfidence, SinglePassConfidence, EnergyConfidence
from confidence.nn import NNDistanceConfidence

In [ ]:
nearest_neighbor_confidence = NNDistanceConfidence(index_type="ivfpq", number_of_neighbors=3)
nearest_neighbor_confidence.fit(embeddings_sampled)
nearest_neighbor_confidence.cuda()
conf_split = SplitConfidence(nearest_neighbor_confidence, EnergyConfidence(), mult=False, b=0.0)
conf_mod = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod,transform_seq,consolidate_method="consolidate_simple")

In [ ]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(transformation_problem, data.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = transformation_problem.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [ ]:
import confidence.model.odin
import importlib

importlib.reload(confidence.model.odin)
from confidence.model.odin import OdinConfidence

from confidence.scaler.calibration import TemperatureCalibrationModule

temp_calibration = TemperatureCalibrationModule(model,per_logit=False).cuda()
temp_calibration.fit(val_loader)
temp_calibration.cuda()

gmm_conf = OdinConfidence(temp_calibration, epsilon=1e-3)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di_no_grad.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [ ]:
import confidence.model.odin
import importlib

importlib.reload(confidence.model.odin)

from confidence.scaler.calibration import TemperatureCalibrationModule
from confidence.base_confidence import MaximumSoftmaxConfidence

temp_calibration = TemperatureCalibrationModule(model,per_logit=False).cuda()
temp_calibration.fit(val_loader)
temp_calibration.cuda()

gmm_conf = SinglePassConfidence(temp_calibration, MaximumSoftmaxConfidence())
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di_no_grad.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [ ]:
from confidence.unsupervised.ml.nn_pytorch import TorchNNConfidence

nn_pytorch = TorchNNConfidence(use_eucldiean_distance=False,per_class=False)
nn_pytorch.fit(embeddings_sampled, classes_sampled)
nn_pytorch.cuda()


conf_split = SplitConfidence(nn_pytorch,EnergyConfidence(), mult=False,b=0.0)
conf_mod_nn_pytorch = SinglePassConfidence(dual_ouput_model,conf_split,index=1)
problem_nn_pytorch = TransformationProblem(conf_mod_nn_pytorch,transform_seq,consolidate_method="consolidate_simple")

In [ ]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_nn_pytorch, data.cuda(),y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_nn_pytorch.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [ ]:
from confidence.gmm import GaussianMixtureConfidence

gmm_conf= GaussianMixtureConfidence(n_components=30,covariance_type="full")
gmm_conf.fit(embeddings_sampled)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

In [ ]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(),y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [ ]:
fake_embeddings = np.random.randn(*embeddings.shape)*2 +embeddings.mean(axis=0)[np.newaxis,:]

In [ ]:
from confidence.isolation import HardIsolationForestConfidence

gmm_conf= HardIsolationForestConfidence()
gmm_conf.fit(embeddings)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

In [ ]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(),y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [ ]:
from confidence.kde import KDEConfidence

gmm_conf = KDEConfidence(bandwidth=0.2,kernel="gaussian")
gmm_conf.fit(embeddings_sampled)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

In [30]:

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.77.


In [31]:
from confidence.unsupervised.ml.osvm import OCSVMPredictiveConfidence

gmm_conf = OCSVMPredictiveConfidence(nu=0.1)
gmm_conf.fit(embeddings_sampled)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")


In [32]:

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.3456.


In [33]:
import confidence.unsupervised.classic.pca
import importlib
importlib.reload(confidence.unsupervised.classic.pca)
from confidence.unsupervised.classic.pca import PCATorchConfidence

gmm_conf = PCATorchConfidence(n_components=32)
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")


In [34]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.6932.


In [35]:
import confidence.gaussian
import importlib
importlib.reload(confidence.gaussian)
from confidence.gaussian import ImageGaussianConfidence

gmm_conf = ImageGaussianConfidence()
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")


In [36]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.7034.


In [37]:
#now tested transfroms that require learning

In [38]:
from confidence.learned.sampling import OnTheFlyNegativeSamplingDataset

In [39]:
negative_sampling_dataset = OnTheFlyNegativeSamplingDataset(torch.tensor(embeddings).cuda())

In [40]:
loader_neg = torch.utils.data.DataLoader(negative_sampling_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [41]:
import confidence.unsupervised.ml.svm
import importlib
importlib.reload(confidence.unsupervised.ml.svm)

In [42]:
# gmm_conf = BinarySVMPredictiveConfidence(kernel="rbf", C=1.0)
# gmm_conf.fit(loader_neg)
# gmm_conf.cuda()
# conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
# gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
# problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

In [43]:
# test_acc_sim = 0
# stop_at = 1
# counter = 0
# for d, (data, target) in enumerate(test_loader):
#     data, target = data.cuda(), target.cuda()
#     res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())
#
#     # Apply the transformation and get predictions
#     x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
#     logits = model(x_transformed2)
#     output = logits.argmax(dim=-1)
#     test_acc_sim += output.eq(target).sum().item()
#     counter += output.shape[0]
#
# test_acc_sim /= counter
#
# print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

In [44]:
import confidence.supervised.ml.energy
import importlib
importlib.reload(confidence.unsupervised.ml.svm)
from confidence.supervised.ml.energy import UnifiedEnergy

In [45]:
energy_model = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Linear(128, 1),
    nn.Sigmoid()
).cuda()

In [46]:
gmm_conf = UnifiedEnergy(energy_model,max_epochs=5)
gmm_conf.fit(loader_neg)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type       | Params | Mode 
----------------------------------------------------
0 | energy_model | Sequential | 131 K  | train
----------------------------------------------------
131 K     Trainable params
0         Non-trainable params
131 K     Total params
0.527     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode
C:\Users\Lindner\PycharmProjects\experiments_rotation\.venv\lib\site-packages\pytorc

Epoch 0:   2%|▏         | 17/938 [00:00<00:08, 104.21it/s, train_loss_step=0.0175]

C:\Users\Lindner\PycharmProjects\experiments_rotation\.venv\lib\site-packages\pytorch_lightning\core\module.py:512: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`


Epoch 4: 100%|██████████| 938/938 [00:06<00:00, 143.92it/s, train_loss_step=9.47e-10, train_loss_epoch=0.000479]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 938/938 [00:06<00:00, 143.89it/s, train_loss_step=9.47e-10, train_loss_epoch=0.000479]


In [47]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.4992.


In [48]:
#test the energy model on embeddings
embeddings_tensor = torch.tensor(embeddings).cuda()
with torch.no_grad():
    for i in range(0, embeddings_tensor.shape[0], batch_size):
        batch = embeddings_tensor[i:i+batch_size]
        energy_scores = energy_model(batch)
        print(f'Batch {i//batch_size}: Energy scores mean: {energy_scores.mean().item()}, std: {energy_scores.std().item()}')
        break

Batch 0: Energy scores mean: 0.9999947547912598, std: 3.554923023330048e-05


In [49]:
#test the energy model on embeddings
embeddings_tensor = torch.tensor(embeddings).cuda()
with torch.no_grad():
    for i in range(0, embeddings_tensor.shape[0], batch_size):
        batch = embeddings_tensor[i:i+batch_size]
        energy_scores = energy_model(batch)
        print(f'Batch {i//batch_size}: Energy scores mean: {energy_scores.mean().item()}, std: {energy_scores.std().item()}')
        break

Batch 0: Energy scores mean: 0.9999947547912598, std: 3.554923023330048e-05


In [50]:
import confidence.unsupervised.ml.autoencoder
import importlib
importlib.reload(confidence.unsupervised.ml.autoencoder)
from confidence.unsupervised.ml.autoencoder import BasicAutoencoderConfidence

In [51]:
encoder_lin = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Linear(128, 16)
).cuda()
decoder_lin = torch.nn.Sequential(
    nn.Linear(16, 128),
    nn.GELU(),
    nn.Linear(128, 512),
    nn.GELU(),
    nn.Linear(512, embeddings.shape[1])
).cuda()

In [52]:
gmm_conf = BasicAutoencoderConfidence(encoder_lin,decoder_lin,max_epochs=5)
gmm_conf.fit(torch.tensor(embeddings).cuda())
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | encoder | Sequential | 133 K  | train
1 | decoder | Sequential | 133 K  | train
-----------------------------------------------
267 K     Trainable params
0         Non-trainable params
267 K     Total params
1.071     Total estimated model params size (MB)
12        Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 1875/1875 [00:08<00:00, 232.24it/s, train_loss_step=0.229, train_loss_epoch=0.235]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 1875/1875 [00:08<00:00, 232.18it/s, train_loss_step=0.229, train_loss_epoch=0.235]


In [53]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.8584.


In [54]:
import confidence.learned.deepsvd
import importlib
importlib.reload(confidence.learned.deepsvd)
from confidence.learned.deepsvd import DeepSVDD

In [55]:
encoder_lin_deep = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 128),
    nn.GELU(),
    nn.Linear(128, 128),
    nn.GELU(),
    nn.Linear(128, 16)
).cuda()

In [56]:
gmm_conf = DeepSVDD(encoder_lin_deep,max_epochs=5,objective="one-class")
gmm_conf.cuda()
gmm_conf.fit(torch.tensor(embeddings).cuda()).cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")

bias terms are not recommended for DeepSVDD, this is not checked here. Similary unbounded activations are prefered


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | encoder | Sequential | 35.1 K | train
-----------------------------------------------
35.1 K    Trainable params
0         Non-trainable params
35.1 K    Total params
0.140     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 469/469 [00:01<00:00, 292.15it/s, train_loss=5.88e-5] 

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 469/469 [00:01<00:00, 291.97it/s, train_loss=5.88e-5]


In [57]:

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.6602.


In [58]:
import confidence.unsupervised.ml.autoencoder
import importlib

importlib.reload(confidence.unsupervised.ml.autoencoder)
from confidence.unsupervised.ml.autoencoder import ContrastiveAutoencoder

encoder_lin2 = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Linear(128, 16)
).cuda()
decoder_lin2 = torch.nn.Sequential(
    nn.Linear(16, 128),
    nn.GELU(),
    nn.Linear(128, 512),
    nn.GELU(),
    nn.Linear(512, embeddings.shape[1]),
).cuda()

gmm_conf = ContrastiveAutoencoder(encoder_lin2, decoder_lin2,max_epochs=5,mode="reconstruction",margin=5).cuda()
gmm_conf.fit(loader_neg)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf2 = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf2, transform_seq, consolidate_method="consolidate_simple")




GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | encoder | Sequential | 133 K  | train
1 | decoder | Sequential | 133 K  | train
-----------------------------------------------
267 K     Trainable params
0         Non-trainable params
267 K     Total params
1.071     Total estimated model params size (MB)
12        Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 938/938 [00:06<00:00, 138.99it/s, train_loss_step=0.266, train_loss_epoch=0.253]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 938/938 [00:06<00:00, 138.97it/s, train_loss_step=0.266, train_loss_epoch=0.253]


In [59]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.854.


In [60]:
import confidence.unsupervised.ml.dis
import importlib

importlib.reload(confidence.unsupervised.ml.dis)
from confidence.unsupervised.ml.dis import SimpleDisConfidence

discriminator = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 2048),
    nn.GELU(),
    nn.Linear(2048, 512),
    nn.GELU(),
    nn.Linear(512, 512),
    nn.GELU(),
    nn.Linear(512, 1)
).cuda()

gmm_conf = SimpleDisConfidence(discriminator,max_epochs=5,noise_std=1,noise_scaling=0.9).cuda()
gmm_conf.fit(loader_neg)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf2 = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf2, transform_seq, consolidate_method="consolidate_simple")




GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type              | Params | Mode 
------------------------------------------------------------
0 | discriminator | Sequential        | 1.6 M  | train
1 | loss_fn       | BCEWithLogitsLoss | 0      | train
------------------------------------------------------------
1.6 M     Trainable params
0         Non-trainable params
1.6 M     Total params
6.306     Total estimated model params size (MB)
9         Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 938/938 [00:06<00:00, 151.49it/s, train/bce_loss_step=0.000118, train/bce_loss_epoch=0.00231]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 938/938 [00:06<00:00, 151.47it/s, train/bce_loss_step=0.000118, train/bce_loss_epoch=0.00231]


In [61]:
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.4192.


In [62]:
#now lets simulate the case where we have true negative samples

In [63]:
#extract true negatives from affine transform dataset training set
true_negatives = []
with torch.no_grad():
    for batch in train_loader_transformed:
        data, target = batch[0], batch[1]
        #use dual output model to get embeddings
        emb, _ = dual_ouput_model(data.cuda())
        true_negatives.append(emb.detach().cpu())
    true_negatives = torch.cat(true_negatives, dim=0)

In [64]:
from confidence.learned.sampling import PreloadedDataset

In [65]:
data_with_negatives = PreloadedDataset(torch.tensor(embeddings).cuda(),true_negatives.cuda())

In [66]:
loader_true_neg = torch.utils.data.DataLoader(data_with_negatives, batch_size=batch_size, shuffle=True, num_workers=0)

In [67]:
import confidence.supervised.ml.energy
import importlib

importlib.reload(confidence.unsupervised.ml.svm)
from confidence.supervised.ml.energy import UnifiedEnergy

energy_model = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Linear(128, 1),
    nn.Sigmoid()
).cuda()
gmm_conf = UnifiedEnergy(energy_model, max_epochs=5)
gmm_conf.fit(loader_true_neg)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf, transform_seq, consolidate_method="consolidate_simple")
test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type       | Params | Mode 
----------------------------------------------------
0 | energy_model | Sequential | 131 K  | train
----------------------------------------------------
131 K     Trainable params
0         Non-trainable params
131 K     Total params
0.527     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 938/938 [00:03<00:00, 256.65it/s, train_loss_step=0.0346, train_loss_epoch=0.0388] 

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 938/938 [00:03<00:00, 256.58it/s, train_loss_step=0.0346, train_loss_epoch=0.0388]
Mean accuracy on the transformed test set with GD: 0.8976.


In [68]:
import confidence.unsupervised.ml.autoencoder
import importlib

importlib.reload(confidence.unsupervised.ml.autoencoder)
from confidence.unsupervised.ml.autoencoder import ContrastiveAutoencoder

encoder_lin2 = torch.nn.Sequential(
    nn.Linear(embeddings.shape[1], 512),
    nn.GELU(),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Linear(128, 16)
).cuda()
decoder_lin2 = torch.nn.Sequential(
    nn.Linear(16, 128),
    nn.GELU(),
    nn.Linear(128, 512),
    nn.GELU(),
    nn.Linear(512, embeddings.shape[1]),
).cuda()

gmm_conf = ContrastiveAutoencoder(encoder_lin2, decoder_lin2, max_epochs=5, mode="reconstruction", margin=5).cuda()
gmm_conf.fit(loader_true_neg)
gmm_conf.cuda()
conf_split = SplitConfidence(gmm_conf, EnergyConfidence(), mult=False, b=0.0)
gmm_conf2 = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
problem_gmm_conf = TransformationProblem(gmm_conf2, transform_seq, consolidate_method="consolidate_simple")

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(problem_gmm_conf, data.cuda(), y=target.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = problem_gmm_conf.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | encoder | Sequential | 133 K  | train
1 | decoder | Sequential | 133 K  | train
-----------------------------------------------
267 K     Trainable params
0         Non-trainable params
267 K     Total params
1.071     Total estimated model params size (MB)
12        Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 938/938 [00:04<00:00, 206.06it/s, train_loss_step=1.180, train_loss_epoch=1.020]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 938/938 [00:04<00:00, 206.01it/s, train_loss_step=1.180, train_loss_epoch=1.020]
Mean accuracy on the transformed test set with GD: 0.9098.


In [69]:
#now with normalization of the data

In [70]:
from confidence.input_transform import InputTransform

In [71]:
input_transform = InputTransform(standardize=True,whiten=False)

In [72]:
input_transform.fit(embeddings)

array([[-0.73057157,  0.        , -0.63794106, ...,  0.45194945,
        -0.31611067,  0.        ],
       [-0.45609194,  0.        ,  1.2530867 , ...,  0.91957545,
        -1.1051301 ,  0.        ],
       [-0.73057157,  0.        , -0.6764973 , ...,  3.2616563 ,
         0.5295318 ,  0.        ],
       ...,
       [-0.01000488,  0.        ,  0.23016408, ...,  0.46789205,
        -1.1051301 ,  0.        ],
       [ 2.6212234 ,  0.        ,  0.05810709, ..., -0.6613256 ,
         0.76427245,  0.        ],
       [-0.73057157,  0.        ,  0.17745927, ..., -0.6613256 ,
        -0.228616  ,  0.        ]], dtype=float32)

In [73]:
nearest_neighbor_confidence = NNDistanceConfidence(index_type="ivfpq", number_of_neighbors=3,input_transform=input_transform)
nearest_neighbor_confidence.fit(embeddings)
nearest_neighbor_confidence.cuda()
conf_split = SplitConfidence(nearest_neighbor_confidence, EnergyConfidence(), mult=False, b=0.0)
conf_mod = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod, transform_seq, consolidate_method="consolidate_simple")

In [74]:

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(transformation_problem, data.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = transformation_problem.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.8288.


In [75]:
from confidence.input_transform import InputTransform

In [76]:
input_transform_whiten = InputTransform(standardize=True,whiten=True)

In [77]:
input_transform_whiten.fit(embeddings)

array([[-6.86188941e-01,  1.38571814e-11, -4.83237658e-01, ...,
        -1.84992202e-01,  3.86730878e-01,  0.00000000e+00],
       [-8.73589147e-01,  3.85939957e-11,  1.53731820e-02, ...,
         3.58680935e-02, -4.39492783e-01,  0.00000000e+00],
       [-1.53362398e-01,  1.06668373e-10,  4.96329978e-01, ...,
         4.55006907e+00, -2.73172651e-02,  0.00000000e+00],
       ...,
       [-1.02247025e-01,  9.09084599e-11, -9.93150188e-01, ...,
         4.04803784e-01, -3.15711803e-01,  0.00000000e+00],
       [ 1.91712803e+00, -1.41599422e-11, -8.41060702e-01, ...,
        -1.25556083e-01,  5.97854382e-01,  0.00000000e+00],
       [ 8.51568509e-01, -4.83750026e-11,  6.69279880e-01, ...,
        -6.52257638e-01, -1.29348983e+00,  0.00000000e+00]])

In [78]:
nearest_neighbor_confidence = NNDistanceConfidence(index_type="ivfpq", number_of_neighbors=3,input_transform=input_transform_whiten)
nearest_neighbor_confidence.fit(embeddings)
nearest_neighbor_confidence.cuda()
conf_split = SplitConfidence(nearest_neighbor_confidence, EnergyConfidence(), mult=False, b=0.0)
conf_mod = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod, transform_seq, consolidate_method="consolidate_simple")

In [79]:

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(transformation_problem, data.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = transformation_problem.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.7776.


In [80]:
from confidence.input_transform import InputTransform

In [81]:
input_transform_whiten_robust = InputTransform(standardize=True,whiten=True)

In [82]:
input_transform_whiten_robust.fit(embeddings)

array([[-6.86188941e-01,  1.38571814e-11, -4.83237658e-01, ...,
        -1.84992202e-01,  3.86730878e-01,  0.00000000e+00],
       [-8.73589147e-01,  3.85939957e-11,  1.53731820e-02, ...,
         3.58680935e-02, -4.39492783e-01,  0.00000000e+00],
       [-1.53362398e-01,  1.06668373e-10,  4.96329978e-01, ...,
         4.55006907e+00, -2.73172651e-02,  0.00000000e+00],
       ...,
       [-1.02247025e-01,  9.09084599e-11, -9.93150188e-01, ...,
         4.04803784e-01, -3.15711803e-01,  0.00000000e+00],
       [ 1.91712803e+00, -1.41599422e-11, -8.41060702e-01, ...,
        -1.25556083e-01,  5.97854382e-01,  0.00000000e+00],
       [ 8.51568509e-01, -4.83750026e-11,  6.69279880e-01, ...,
        -6.52257638e-01, -1.29348983e+00,  0.00000000e+00]])

In [83]:
nearest_neighbor_confidence = NNDistanceConfidence(index_type="ivfpq", number_of_neighbors=3,input_transform=input_transform_whiten_robust)
nearest_neighbor_confidence.fit(embeddings)
nearest_neighbor_confidence.cuda()
conf_split = SplitConfidence(nearest_neighbor_confidence, EnergyConfidence(), mult=False, b=0.0)
conf_mod = SinglePassConfidence(dual_ouput_model, conf_split, index=1)
transformation_problem = TransformationProblem(conf_mod, transform_seq, consolidate_method="consolidate_simple")

In [84]:

test_acc_sim = 0
stop_at = 1
counter = 0
for d, (data, target) in enumerate(test_loader):
    data, target = data.cuda(), target.cuda()
    res = di.optimize(transformation_problem, data.cuda())

    # Apply the transformation and get predictions
    x_transformed2 = transformation_problem.transform(data.cuda(), res[0])
    logits = model(x_transformed2)
    output = logits.argmax(dim=-1)
    test_acc_sim += output.eq(target).sum().item()
    counter += output.shape[0]

test_acc_sim /= counter

print(f'Mean accuracy on the transformed test set with GD: {test_acc_sim}.')

Mean accuracy on the transformed test set with GD: 0.7892.
